# 02 — Trusted Transformation

Transformacion de Raw a Trusted:
1. Renombrar columnas a snake_case
2. Calcular `trip_duration_minutes`
3. Validar registros (pickup < dropoff, distance > 0, fare > 0, outliers)
4. Manejo de nulos con defaults conservadores
5. JOIN con Taxi Zones (zona y borough de pickup)
6. Separar en `_clean` y `_rejected`

In [ ]:
%run ../config/pipeline_config

In [ ]:
import logging
import time
import json
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s - %(message)s")
logger = logging.getLogger("trusted_transformation")

start_time = time.time()
metrics = {"stage": "02_trusted_transformation"}

## Leer tablas Raw

In [ ]:
df = spark.table(RAW_TAXI_TABLE)
df_zones = spark.table(RAW_ZONES_TABLE)

total_raw = df.count()
logger.info(f"Registros leidos de raw.yellow_taxi_trips: {total_raw:,}")
logger.info(f"Registros leidos de raw.taxi_zone_lookup: {df_zones.count()}")

metrics["total_raw"] = total_raw

## Renombrar columnas

In [ ]:
for old_name, new_name in COLUMN_RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

logger.info(f"Columnas renombradas: {COLUMN_RENAME_MAP}")

## Duracion del viaje

In [ ]:
# Se calcula antes de validar porque las reglas de outliers usan la duracion
df = df.withColumn(
    "trip_duration_minutes",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

## Validacion y etiquetado de rechazos

Cada registro recibe la **primera regla que falla** como `rejection_reason`. Esto evita doble conteo en el KPI 3.

| Regla | Condicion de rechazo |
|-------|---------------------|
| `invalid_time_order` | pickup >= dropoff |
| `invalid_distance` | distance <= 0 |
| `invalid_fare` | fare <= 0 |
| `negative_total` | total < 0 |
| `outlier_distance` | distance > 200 mi |
| `outlier_fare` | fare > 500 USD |
| `outlier_duration_high` | duracion > 300 min |
| `outlier_duration_low` | duracion < 1 min |
| `outlier_passengers` | pasajeros > 6 |
| `out_of_range_date` | fuera de enero 2023 |

In [ ]:
df = df.withColumn(
    "rejection_reason",
    F.when(
        F.col("tpep_pickup_datetime") >= F.col("tpep_dropoff_datetime"), F.lit("invalid_time_order")
    ).when(
        F.col("trip_distance") <= 0, F.lit("invalid_distance")
    ).when(
        F.col("fare_amount") <= 0, F.lit("invalid_fare")
    ).when(
        F.col("total_amount") < MIN_TOTAL_AMOUNT, F.lit("negative_total")
    ).when(
        F.col("trip_distance") > MAX_TRIP_DISTANCE_MILES, F.lit("outlier_distance")
    ).when(
        F.col("fare_amount") > MAX_FARE_AMOUNT, F.lit("outlier_fare")
    ).when(
        F.col("trip_duration_minutes") > MAX_TRIP_DURATION_MINUTES, F.lit("outlier_duration_high")
    ).when(
        F.col("trip_duration_minutes") < MIN_TRIP_DURATION_MINUTES, F.lit("outlier_duration_low")
    ).when(
        F.col("passenger_count") > MAX_PASSENGER_COUNT, F.lit("outlier_passengers")
    ).when(
        (F.year("tpep_pickup_datetime") != EXPECTED_YEAR) |
        (F.month("tpep_pickup_datetime") != EXPECTED_MONTH),
        F.lit("out_of_range_date")
    )
)

## Separar validos vs rechazados

In [ ]:
# Separar registros
df_rejected = df.filter(F.col("rejection_reason").isNotNull())
df_clean = df.filter(F.col("rejection_reason").isNull()).drop("rejection_reason")

rejected_count = df_rejected.count()
clean_count = df_clean.count()

logger.info(f"Registros validos: {clean_count:,} ({round(clean_count/total_raw*100, 1)}%)")
logger.warning(f"Registros rechazados: {rejected_count:,} ({round(rejected_count/total_raw*100, 1)}%)")

# Detalle de rechazos por regla
rejection_breakdown = (
    df_rejected.groupBy("rejection_reason")
    .count()
    .orderBy(F.col("count").desc())
)
rejection_breakdown.show(truncate=False)

# Guardar breakdown como dict para metricas
metrics["rejected_total"] = rejected_count
metrics["clean_total"] = clean_count
metrics["rejection_breakdown"] = {
    row["rejection_reason"]: row["count"] 
    for row in rejection_breakdown.collect()
}

## Nulos

In [ ]:
null_counts = {}
for col_name in ["passenger_count", "ratecode_id", "payment_type", "store_and_fwd_flag", "congestion_surcharge", "airport_fee"]:
    null_count = df_clean.filter(F.col(col_name).isNull()).count()
    if null_count > 0:
        null_counts[col_name] = null_count
        logger.warning(f"Nulos en '{col_name}': {null_count:,}")

df_clean = df_clean.fillna({
    "passenger_count": 1,
    "ratecode_id": 99,
    "payment_type": 0,
    "store_and_fwd_flag": "N",
    "congestion_surcharge": 0.0,
    "airport_fee": 0.0,
})

metrics["nulls_replaced"] = null_counts
logger.info(f"Nulos reemplazados: {null_counts}")

## JOIN con Taxi Zones

In [ ]:
df_zones_pickup = df_zones.select(
    F.col("LocationID"),
    F.col("Zone").alias("pickup_zone"),
    F.col("Borough").alias("pickup_borough"),
    F.col("service_zone").alias("pickup_service_zone")
)

df_clean = df_clean.join(
    df_zones_pickup,
    df_clean["pickup_location_id"] == df_zones_pickup["LocationID"],
    "left"
).drop("LocationID")

unmapped = df_clean.filter(F.col("pickup_zone").isNull()).count()
if unmapped > 0:
    logger.warning(f"Registros sin zona de pickup mapeada: {unmapped:,}")
else:
    logger.info("Todos los registros tienen zona de pickup mapeada.")

metrics["unmapped_zones"] = unmapped

## Auditoria

In [ ]:
df_clean = df_clean.withColumn("ingestion_timestamp", F.current_timestamp())
df_clean = df_clean.withColumn("source_file", F.lit("yellow_tripdata_2023-01.parquet"))

logger.info(f"Total columnas: {len(df_clean.columns)}")
df_clean.printSchema()

## Escritura a Delta

In [ ]:
try:
    df_clean.write.format("delta").mode("overwrite").saveAsTable(TRUSTED_CLEAN_TABLE)
    spark.sql(f"COMMENT ON TABLE {TRUSTED_CLEAN_TABLE} IS 'Viajes validados y enriquecidos con zona de pickup.'")

    props = ", ".join([f"'{k}' = '{v}'" for k, v in TABLE_PROPERTIES["trusted"].items()])
    spark.sql(f"ALTER TABLE {TRUSTED_CLEAN_TABLE} SET TBLPROPERTIES ({props})")

    spark.sql(f"OPTIMIZE {TRUSTED_CLEAN_TABLE} ZORDER BY (pickup_location_id, tpep_pickup_datetime)")
    logger.info(f"'{TRUSTED_CLEAN_TABLE}': {clean_count:,} registros.")
except Exception as e:
    logger.error(f"Error escribiendo trusted clean: {e}")
    raise

try:
    df_rejected.write.format("delta").mode("overwrite").saveAsTable(TRUSTED_REJECTED_TABLE)
    spark.sql(f"COMMENT ON TABLE {TRUSTED_REJECTED_TABLE} IS 'Registros rechazados con rejection_reason.'")

    props = ", ".join([f"'{k}' = '{v}'" for k, v in TABLE_PROPERTIES["trusted"].items()])
    spark.sql(f"ALTER TABLE {TRUSTED_REJECTED_TABLE} SET TBLPROPERTIES ({props})")

    spark.sql(f"OPTIMIZE {TRUSTED_REJECTED_TABLE} ZORDER BY (rejection_reason)")
    logger.info(f"'{TRUSTED_REJECTED_TABLE}': {rejected_count:,} registros.")
except Exception as e:
    logger.error(f"Error escribiendo trusted rejected: {e}")
    raise

## Resumen

In [ ]:
elapsed = round(time.time() - start_time, 2)
metrics["duration_seconds"] = elapsed
metrics["status"] = "SUCCESS"

logger.info(f"Trusted transformation completada en {elapsed}s.")
logger.info(f"Resumen: {total_raw:,} raw -> {clean_count:,} clean + {rejected_count:,} rejected")
logger.info(f"Tasa de descarte: {round(rejected_count/total_raw*100, 2)}%")

print("\n" + "="*60)
print(json.dumps(metrics, indent=2, default=str))
print("="*60)

try:
    dbutils.notebook.exit(json.dumps(metrics, default=str))
except NameError:
    pass